# Generate AI Reviews

This notebook generates both NCEMS-criteria reviews and novelty reviews for a single experimental condition, then builds `review_scores_wide.csv` for downstream analysis.

Run this notebook after `compare_proposals_rephrased.ipynb`, because novelty review generation depends on `proposal_lit_neighbors.json`.

## Condition Configuration

In [ ]:
# Auto-managed by src/run_condition.py
CONDITION = 'minimal'


## Setup and Path Checks

In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / 'src'
AI_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'ai-proposals' / 'rephrased' / CONDITION
HUMAN_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'human-proposals' / 'rephrased' / CONDITION
LIT_NEIGHBORS_PATH = PROJECT_ROOT / 'results' / 'tables' / 'rephrased' / CONDITION / 'proposal_lit_neighbors.json'
NCEMS_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / CONDITION / 'ncems_criteria'
NOVELTY_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / CONDITION / 'novelty'
REVIEW_SCORES_PATH = PROJECT_ROOT / 'results' / 'tables' / 'rephrased' / CONDITION / 'review_scores_wide.csv'

print(f'Condition: {CONDITION}')
print(f'Project root: {PROJECT_ROOT}')
print(f'AI rephrased dir: {AI_REPHRASED_DIR}')
print(f'Human rephrased dir: {HUMAN_REPHRASED_DIR}')
print(f'Literature neighbors path: {LIT_NEIGHBORS_PATH}')

if not AI_REPHRASED_DIR.exists():
    raise FileNotFoundError(f'Missing AI rephrased proposals directory: {AI_REPHRASED_DIR}')
if not HUMAN_REPHRASED_DIR.exists():
    raise FileNotFoundError(f'Missing human rephrased proposals directory: {HUMAN_REPHRASED_DIR}')
if not LIT_NEIGHBORS_PATH.exists():
    raise FileNotFoundError(
        f'Missing literature neighbors file: {LIT_NEIGHBORS_PATH}. '
        'Run compare_proposals_rephrased.ipynb first.'
    )

print('✓ Required proposal and literature-neighbor inputs are present')


## Generate NCEMS-Criteria Reviews

In [ ]:
subprocess.check_call([
    sys.executable,
    str(SCRIPTS_DIR / 'generate_reviews_ncems_criteria.py'),
    '--condition',
    CONDITION,
], cwd=PROJECT_ROOT)

latest_ncems = sorted(NCEMS_REVIEWS_DIR.glob('ncems_reviews_*.json'))[-1]
print(f'✓ NCEMS reviews written to: {latest_ncems}')


## Generate Novelty Reviews

In [ ]:
subprocess.check_call([
    sys.executable,
    str(SCRIPTS_DIR / 'generate_reviews_novelty.py'),
    '--condition',
    CONDITION,
], cwd=PROJECT_ROOT)

latest_novelty = sorted(NOVELTY_REVIEWS_DIR.glob('novelty_reviews_*.json'))[-1]
print(f'✓ Novelty reviews written to: {latest_novelty}')


## Build Review Score Table

In [ ]:
subprocess.check_call([
    sys.executable,
    str(SCRIPTS_DIR / 'build_review_scores_wide.py'),
    '--condition',
    CONDITION,
], cwd=PROJECT_ROOT)

print(f'✓ Review score table written to: {REVIEW_SCORES_PATH}')


## Output Summary

In [ ]:
summary = {
    'condition': CONDITION,
    'latest_ncems_reviews': str(sorted(NCEMS_REVIEWS_DIR.glob('ncems_reviews_*.json'))[-1]),
    'latest_novelty_reviews': str(sorted(NOVELTY_REVIEWS_DIR.glob('novelty_reviews_*.json'))[-1]),
    'review_scores_wide': str(REVIEW_SCORES_PATH),
}

print(json.dumps(summary, indent=2))
